# [ 실습4 ] HuggingFace 첫 감정분석 모델

> AI기반 실시간 대화 분석 및 맥락정리 기술개발 R&D — HuggingFace 입문

## 실습 목표

1. `pipeline()` API로 HuggingFace 첫 모델 동작 확인
2. 모델 내부 구조 살펴보기 (DistilBERT 아키텍처)
3. Sarcasm / 혼합 감정으로 모델 한계 체감
4. 한국어 감정분석 모델로 교체
5. 회의 데이터에 적용 — 강요된 동의 컨셉 연결

## 환경

| 항목 | 버전 |
|------|------|
| Python | 3.11.15 |
| transformers | 5.9.0 |
| torch | 2.11.0+cu128 |
| GPU | RTX 5070 (Blackwell, sm_120) |

transformers 5.9.0 검증 완료. `pipeline()` API는 4.x에서 5.x로 넘어오면서 기본 호출 방식은 변화가 거의 없음. 단, `device="cuda"` 명시 필요 (자동 GPU 할당 안 됨).

## 연구 컨텍스트

XR 환경 다자간 대화에서 **표면과 속내의 괴리** 탐지가 최종 목표.

- "네 좋아요" + 심박 스트레스 → 강요된 동의
- 칭찬 + 팔짱 + 눈 굴림 → Sarcasm

본 실습의 감정분석은 텍스트 모달리티의 출발점에 해당.

In [2]:
# 환경 확인 — 이 노트북이 어떤 환경에서 실행됐는지 영구 기록
import sys
import torch
import transformers

print(f"Python:       {sys.version.split()[0]}")
print(f"transformers: {transformers.__version__}")
print(f"torch:        {torch.__version__}")
print(f"CUDA 사용:    {torch.cuda.is_available()}")
print(f"GPU:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

Python:       3.11.15
transformers: 5.9.0
torch:        2.11.0+cu128
CUDA 사용:    True
GPU:          NVIDIA GeForce RTX 5070


In [3]:
# HuggingFace pipeline 생성
# task: sentiment-analysis (text-classification의 별칭)
# device: cuda → RTX 5070 사용 (명시 안 하면 CPU로 돌아감)
from transformers import pipeline # HuggingFace의 transformers 라이브러리에서 pipeline이라는 함수를 가져옴

classifier = pipeline("sentiment-analysis", device="cuda")
# pipeline()의 첫 번째 인자: task: 어떤 모델을 가져올지 명시(위 코드는 감정분석 모델)
# cuda: nvida gpu -> 이것을 명시해야 GPU를 사용
# 변환된 객체를 classifier 변수에 저장

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

모델을 명시하지 않으면 기본값을 골라줌.

실제 운용/논문 코드에서는 명시해야 할듯.

```python
pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    revision="714eb0f",   # 특정 버전 고정
    device="cuda",
)
```

In [6]:
# 첫 추론 - 영어 한 문장을 분류해보기
result = classifier("I love this movie!")
print(result)

# 이렇게 classifier()을 한 번 호출하면
# 1: 토크나이저가 문자열 -> 토큰 ID 리스트로 변환
# 2: 토큰 ID를 텐서로 변환 후 GPU 이동
# 3: DistilBERT 모델이 forward pass 실행(6개의 transformer 레이어 통과)
# 4: 마지막 토큰 출력에서 분류 점수 계산(ex..POSITIVE = 0.9998775720596313 / NEGATIVE = 0.0001224279403687 에서 큰 거 선택)
# 5: softmax로 확률화 -> 가장 높은 라벨 선택
# 6: 사람이 읽기 좋은 dict 형태로 출력

[{'label': 'POSITIVE', 'score': 0.9998775720596313}]


In [ ]:
# 여러 문장을 한 번에 분류 - 모델의 패턴과 한계 동시 관찰
sentences = [
    "I love this movie!",                              # 명백한 긍정
    "This is the worst film ever.",                    # 명백한 부정
    "It was okay, I guess.",                           # 애매한 중립
    "Oh great, another Monday morning.",               # Sarcasm (비꼼) -> 사실은 부정인데 긍정처럼 보이는 문장
    "The food was amazing but service was terrible.",  # 혼합 감정
]

results = classifier(sentences)

# 그 결과를 보기 좋게 출력
for sentence, result in zip(sentences, results):
    label = result['label']
    score = result['score']
    print(f"[{label:8s} {score:.4f}]  {sentence}")

# 5개 문장 입력하면 토크나이저가 5개 모두 토큰화 + 패딩(길이 맞춤)
# GPU로 5개를 한 번에 전송
# DistilBERT가 5개를 병렬 처리(GPU 장점) 후 5개 결과를 동시에 받음 -> GPU는 병렬 연산에 강해서 입력 수가 늘어나도 처리 시간이 비례해서 늘어나지 않음

[POSITIVE 0.9999]  I love this movie!
[NEGATIVE 0.9997]  This is the worst film ever.
[POSITIVE 0.9997]  It was okay, I guess.
[POSITIVE 0.9983]  Oh great, another Monday morning.
[POSITIVE 0.6300]  The food was amazing but service was terrible.


In [10]:
# 결과 해석
# - 지금 사용 중인 모델: distilbert/distilbert-base-uncased-finetuned-sst-2-english -> 이진 분류 모델
# - 3번째 결과 보면 "애매한 중립" 문장을 강력한 긍정 문장으로 파악 -> 이진 분류의 한계점(중립 옵션이 없음 + 높은 홧신으로 답함)
# - 4번째 결과 보면 사회적인 의미로 부정 문장인데 다눈히 문장 속 단어만 파악하고 긍정 문장으로 파악 -> 즉 이러한 한계점이 바로 멀티모달이 필요한 이유
# - 5번째 결과 보면 이전까지는 모두 높은 score 점수로 예측을 했는데 0.6300이라는 낮은 스코어로 예측 -> 모델이 헷갈렸다는 징조

In [ ]:
# 챌린지 2: 영어 Sarcasm 명장 3개 만들기
# 의도: 표면 단어와 실제 의미의 괴리를 만들어서 모델이 속는지 확인

# 내가 생성한 문장
my_sarcasm = [
    "Wonderful, my flight got delayed by 5hours", # 좋네, 내 비행기 5시간 지연됐어
    "What a brilliant idea to schedule a meeting at 6 AM", # 새벽 6시에 회의를 잡다니, 정말 똑똑한 생각이네
    "Just what I needed, more emails to answer", # 내가 딱 필요했던 거네, 답장할 이메일이 더 늘었어
]

# 분류 실행
my_results = classifier(my_sarcasm)

# 결과 출력
for sentence, result in zip(my_sarcasm, results):
    label = result['label']
    score = result['score']
    print(f"{label:8s} {score:.4f} {sentence}")

POSITIVE 0.9999 Wonderful, my flight got delayed by 5hours
NEGATIVE 0.9997 What a brilliant idea to schedule a meeting at 6 AM
POSITIVE 0.9997 Just what I needed, more emails to answer


In [ ]:
# 결과 분석
# 직접 만든 sarcasm 3개 중 2개 속음, 1개 잡힘.

# 잡힌 것: "What a brilliant idea to schedule a meeting at 6 AM"
# → "What a [긍정] idea" 패턴이 학습 데이터에 있었던 듯
# → 모델은 "Sarcasm 이해"가 아닌 "Sarcasm 패턴 암기"

# 속은 것: "Wonderful" 시작 / "Just what I needed" 클리셰
# → 자유 형식 sarcasm은 표면 단어에 압도됨
# → 사회적 맥락 (5시간 지연=짜증)은 텍스트로만 못 잡음

# 연구 의의: 텍스트 단독은 데이터 분포에 의존
# 멀티모달(표정, 심박)이 데이터-독립적 객관 신호 제공

In [17]:
# 영어 전용 DistilBERT에 한국어 문장을 넣으면??
# 의도: 영어 모델의 한국어 처리 한계 체험

ko_sentences = [
    "이 영화 진짜 최고였어요!",
    "최악의 영화. 시간 낭비.",
    "그저 그랬어요.",
    "네 좋아요.", # 우리 연구의 핵심 표현
    "아 정말요, 너무 좋네요.", # 한국어 sarcasm
]

# 기존 영어 모델(classifier)에 한국어 던지기
en_model_ko_results = classifier(ko_sentences)

# 결과 출력
for sentence, result in zip(ko_sentences, en_model_ko_results):
    label = result['label']
    score = result['score']
    print(f"[{label:8s} {score:.4f}]  {sentence}")

[POSITIVE 0.9897]  이 영화 진짜 최고였어요!
[POSITIVE 0.7239]  최악의 영화. 시간 낭비.
[POSITIVE 0.8212]  그저 그랬어요.
[POSITIVE 0.8516]  네 좋아요.
[POSITIVE 0.6618]  아 정말요, 너무 좋네요.


In [ ]:
# 결과 분석

# 영어 모델에 한국어 5문장 → 5개 모두 POSITIVE.
# random(2~3개 POSITIVE)보다 더 편향된 결과.

# 원인: 한국어 토큰이 영어 vocab에 없어서 [UNK] 처리,
# 모델이 학습된 평균 응답으로 기울어짐.

# 연구 의의: 우리 회의 데이터(전부 한국어) 분석은 한국어 모델이 필수.
# 영어 모델은 평가조차 불가능.

In [21]:
# 한국어 감정 분석 전용 모델로 새 pipeline 생성
# 모델: matthewburke/korean_sentiment (한국어 감정분석으로 직접 파인튜닝됨)

# 모델의 라벨 매핑 확인
print("id2label:", ko_classifier.model.config.id2label)
print("label2id:", ko_classifier.model.config.label2id)

ko_classifier = pipeline(
    "sentiment-analysis",
    model= "matthewburke/korean_sentiment", # 한국어 감정 분류 모델 명시
    device= "cuda" # GPU 사용 명시
)

ko_classifier.model.config.id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
ko_classifier.model.config.label2id = {'NEGATIVE': 0, 'POSITIVE': 1}

ko_results = ko_classifier(ko_sentences)

for sentence, result in zip(ko_sentences, ko_results):
    label = result['label']
    score = result['score']
    print(f"[{label:8s} {score:.4f}]  {sentence}")


# pipeline() 호출되면 첫 시도에만 모델이 다운로드 되고 한국어 토크나이저 로드되고 분류 점수 계산 + softmax로 인해 확률화 된 후 출력


id2label: {0: 'NEGATIVE', 1: 'POSITIVE'}
label2id: {'NEGATIVE': 0, 'POSITIVE': 1}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[POSITIVE 0.9738]  이 영화 진짜 최고였어요!
[NEGATIVE 0.9660]  최악의 영화. 시간 낭비.
[NEGATIVE 0.9554]  그저 그랬어요.
[POSITIVE 0.8841]  네 좋아요.
[POSITIVE 0.9707]  아 정말요, 너무 좋네요.


In [ ]:
# 결과 분석

# 새 모델 사용 시, 라벨 검증을 항상 먼저 하는 것이 좋을듯
# print("id2label:", model.config.id2label)
# print("label2id:", model.config.label2id)

# 좋은 예 (영어 DistilBERT):
# id2label: {0: 'NEGATIVE', 1: 'POSITIVE'}
# → 모델 자체가 "0번은 부정, 1번은 긍정" 이라고 명시함

# 이 모델 (matthewburke/korean_sentiment):
# id2label: {0: 'LABEL_0', 1: 'LABEL_1'}
# → 모델 자체는 라벨 의미를 모름. 사용자가 추측해야 함

# 그래서 라벨이 매핑이 되어있지 않다면 사람 친화적으로 매핑할 필요가 있음
# ko_classifier.model.config.id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
# ko_classifier.model.config.label2id = {'NEGATIVE': 0, 'POSITIVE': 1}

In [ ]:
# 회의 데이터 만들고 적용해보기
# 회의 데이터 생성
import pandas as pd

meeting_data = pd.DataFrame({
    'time':      ['10:00', '10:01', '10:02', '10:03', '10:04', '10:05'],
    'speaker':   ['노태경', '서수화', '박태근', '기예진', '이연재', '노태경'],
    'utterance': [
        '이번 일정 무리하지 않을까요?',
        '괜찮아요, 충분히 가능합니다.',
        '네 좋아요.', # 강요된 동의 의심
        '태근이도 동의했네요',  
        '아 정말요, 너무 좋네요.', # Sarcasm 의심
        '그럼 그렇게 진행합시다.',
    ],
    'heart_rate': [98, 68, 95, 71, 102, 85], # bpm
})

display(meeting_data)

,time,speaker,utterance,heart_rate
0,10:00,노태경,이번 일정 무리하지 않을까요?,98
1,10:01,서수화,"괜찮아요, 충분히 가능합니다.",68
2,10:02,박태근,네 좋아요.,95
3,10:03,기예진,태근이도 동의했네요,71
4,10:04,이연재,"아 정말요, 너무 좋네요.",102
5,10:05,노태경,그럼 그렇게 진행합시다.,85


In [ ]:
# 회의 발화에 한국어 감정 분석 적용
def analyze_sentiment(text):
    result = ko_classifier(text)[0]
    return pd.Series([result['label'], result['score']])

meeting_data[['sentiment', 'score']] = meeting_data['utterance'].apply(analyze_sentiment)
# .apply(analyze_sentiment): 각 행마다 함수 적용 -> meeting_data['utterance']의 각 행마다 analyze_sentiment 함수 적용 -> 각 함수가 label, score 반환 -> sentiment와 score로 변환해서 컬럼 추가

display(meeting_data)

,time,speaker,utterance,heart_rate,sentiment,score
0,10:00,노태경,이번 일정 무리하지 않을까요?,98,NEGATIVE,0.786057
1,10:01,서수화,"괜찮아요, 충분히 가능합니다.",68,POSITIVE,0.962655
2,10:02,박태근,네 좋아요.,95,POSITIVE,0.884142
3,10:03,기예진,태근이도 동의했네요,71,NEGATIVE,0.685319
4,10:04,이연재,"아 정말요, 너무 좋네요.",102,POSITIVE,0.970691
5,10:05,노태경,그럼 그렇게 진행합시다.,85,POSITIVE,0.862547


In [29]:
# 결과를 분석해보니 발화 텍스트만 보고 긍정/부정을 판단하는 것은 역시 한계가 존재 -> 멀티 모달의 필요성 

In [ ]:
suspicious = meeting_data[
    (meeting_data['sentiment'] == 'POSITIVE') & 
    (meeting_data['heart_rate'] >= 90)
]
# suspicious = sentiment가 POSIIVE인데 heart_rate가 90이상인 것만 

display(suspicious)

,time,speaker,utterance,heart_rate,sentiment,score
2,10:02,박태근,네 좋아요.,95,POSITIVE,0.884142
4,10:04,이연재,"아 정말요, 너무 좋네요.",102,POSITIVE,0.970691


## 실습 종합 정리

### 1. 분석한 회의 데이터

6명의 화자, 6개 발화. 의도적으로 "표면 동의 + 심박 스트레스" 패턴 포함.

### 2. 한국어 감정분석 모델 결과

`matthewburke/korean_sentiment` 적용 결과 정확도 한계 확인:

| 케이스 | 결과 | 평가 |
|--------|------|------|
| 명백한 긍정/부정 | 잘 잡음 | OK |
| 질문/우려 표현 | 단어 단위로 부정 분류 | 의도 못 잡음 |
| 사실 진술 | 부정으로 헷갈림 | 신호 약함 |
| Sarcasm | POSITIVE 0.97 강한 확신 | 완전히 속음 |

### 3. 의심 발화 검출 (감정 + 심박 결합)

조건: sentiment == POSITIVE AND heart_rate >= 90

| 화자 | 발화 | 심박 | sentiment | 의심 |
|------|------|------|-----------|------|
| 박태근 | "네 좋아요" | 95 | POSITIVE 0.88 | 강요된 동의 |
| 이연재 | "아 정말요, 너무 좋네요" | 102 | POSITIVE 0.97 | Sarcasm |

### 4. 핵심 발견

**텍스트 단독으로는 두 발화 모두 "동의/긍정"으로 분류됨.**
**심박 결합 시 두 발화 모두 "신뢰 못 함, 의심"으로 재해석됨.**

같은 발화에 대해 완전히 반대 결론 -> 멀티모달이 필요한 이유

### 5. 우리 연구로의 시사점

- 감정분석 모델의 본질적 한계: 표면 단어에 의존, 의도/사회적 맥락 못 잡음
- 한국어 모델로 바꿔도 sarcasm 한계는 동일 (언어 무관 문제)
- 생리신호(심박)는 학습 데이터에 비의존적, 객관 지표
- 단일 모달리티 분석은 출발점이지 답이 아님
- 1차년도 핵심 LLaMA-3 + LoRA도 텍스트 기반 → 동일 한계 예상
- 멀티모달 융합 (텍스트 + 표정 + 심박)이 연구의 본질적 답